# Pillar3c: Phase 3 counterfactual distillation

## Goal

Lift pillar3b's policy-only floor (P5 1,576 → ≥2,500, <1000 rate 2.5% → ≤1.5%) without regressing the mean (17,255) more than 5%. Browser/WASM deployment target.

## Method (locked 2026-05-24 after two rounds of ChatGPT peer review)

Add a listwise margin auxiliary loss on 528 stationary-boundary anchors where pillar3b's top-1 is genuinely floor-suboptimal (Phase 2 mining: top-1 wins floor in only 18% of stationary states).

**Loss:** per anchor, push floor-winner logit above stored top1 + clean-other losers by `margin=0.25`. Weight 1.0 (top1 pair), 0.5 (clean others).

**Filter:** anchor passes iff `die_diff[top1, winner] ≥ 1/24 OR leave_diff[top1, winner] ≥ 2/24` → 528 anchors (~3,883 pairs). Per-pair filter applied identically.

**Schedule:** λ warmup 0 → 0.20 over first 2 epochs. Aux batch 256 per main step (corpus cycles every ~2 main steps).

**Preflight metrics** every 200 main steps:
- `stored_top1_flip_rate`: fraction of anchors where logit[winner] > logit[stored_top1]. Baseline = 0.002. Step-500 target: > 0.05 (else abort).
- `all_clean_loser_margin_rate` @ margin 0.25. Baseline = 0.438.

**Abort gate:** auto-aborts if step-500+ preflight reports `top1_flip_rate < 0.05`.

## Required Drive uploads (`MyDrive/alphatrain/`)

1. `colorlines_pillar3c.tar.gz` — code archive including the new Phase 3 files
2. `v13_pillar3a.pt.gz` — V13 tensor (already on Drive from pillar3b training)
3. `pillar3b_epoch_20.pt` — warm-start checkpoint (already on Drive)

**Build the tarball locally first** (from repo root):
```bash
tar czf colorlines_pillar3c.tar.gz \
    --exclude='**/__pycache__' \
    alphatrain/counterfactual.py \
    alphatrain/train_path_b.py \
    alphatrain/model.py \
    alphatrain/dataset.py \
    alphatrain/observation.py \
    alphatrain/evaluate.py \
    alphatrain/data/stationary_counterfactuals_v1.pt \
    scripts/preflight_phase3.py \
    game/
```

Then upload `colorlines_pillar3c.tar.gz` to `MyDrive/alphatrain/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_pillar3c.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3c.tar.gz

# Warm-start checkpoint
os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt',
            '/content/alphatrain/data/pillar3b_epoch_20.pt')
print(f'pillar3b: {os.path.getsize("/content/alphatrain/data/pillar3b_epoch_20.pt")/1e6:.0f} MB')

# Counterfactual corpus (extracted from tarball)
cf_path = '/content/alphatrain/data/stationary_counterfactuals_v1.pt'
assert os.path.exists(cf_path), 'counterfactual corpus missing from tarball'
print(f'counterfactuals: {os.path.getsize(cf_path)/1e6:.1f} MB')

# V13 training tensor (~5 GB after decompression)
t0 = time.time()
!gzip -dc {DRIVE}/v13_pillar3a.pt.gz > /content/alphatrain/data/v13_pillar3a.pt
print(f'V13 tensor: {os.path.getsize("/content/alphatrain/data/v13_pillar3a.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

## Baseline preflight on pillar3b

Confirm the metrics match the M5 local baseline before burning ~12h of training compute:
- `stored_top1_flip_rate ≈ 0.002`
- `all_clean_loser_margin_rate @ 0.25 ≈ 0.438`
- median `logit[winner] - logit[stored_top1] ≈ -1.76`

If any of these are way off, abort and investigate before training.

In [ ]:
%cd /content
!PYTHONPATH=. python scripts/preflight_phase3.py \
    --model alphatrain/data/pillar3b_epoch_20.pt \
    --records alphatrain/data/stationary_counterfactuals_v1.pt \
    --margin 0.25

## Train pillar3c (λ=0.20, primary run)

~12h on G4 / L4. Checkpoints copied to Drive every epoch via `--copy-to`.

**Watch the preflight prints** during training:
- Step 500: `top1_flip` should be visibly > 0.05 (auto-aborts otherwise)
- By epoch 3-5: `top1_flip` should reach 0.30+
- End: `top1_flip` ideally 0.50+ AND main val_loss within 5% of pillar3b's pillar3b val_loss (V13 distillation must stay anchored)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
    --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --aux-counterfactual alphatrain/data/stationary_counterfactuals_v1.pt \
    --aux-lambda 0.20 --aux-margin 0.25 \
    --aux-batch-size 256 --aux-warmup-epochs 2.0 \
    --aux-preflight-every 200 \
    --aux-abort-flip-rate 0.05 --aux-abort-after-step 500 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3c_best.pt \
    --save-dir /content/checkpoints/pillar3c 2>&1 | tee /content/pillar3c_train.log

In [ ]:
# Extract preflight trajectory from log for at-a-glance review
!grep -E 'preflight|Train: loss|val:' /content/pillar3c_train.log

In [ ]:
# Copy all epoch checkpoints and the training log to Drive
import glob, shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar3c/epoch_*.pt')):
    dst = f'{DRIVE}/pillar3c_{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print(f'copied {os.path.basename(f)}')
shutil.copy('/content/pillar3c_train.log', f'{DRIVE}/pillar3c_train.log')
print('log copied')

## Optional: λ ablation (only if primary run is acceptable)

Don't run these unless λ=0.20 produces a checkpoint hitting the decision gate (P5 ≥ 2200 AND <1000 ≤ 1.5% AND mean within 5%). Each is another ~12h.

If primary run looks too weak (top1_flip never crosses 0.30 by epoch 8) → λ=0.40.
If primary run regresses mean too much → λ=0.10.

In [ ]:
# # Stronger aux: λ=0.40
# %cd /content
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
#     --tensor-file alphatrain/data/v13_pillar3a.pt \
#     --amp --compile \
#     --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
#     --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
#     --target-temperature 0.5 \
#     --aux-counterfactual alphatrain/data/stationary_counterfactuals_v1.pt \
#     --aux-lambda 0.40 --aux-margin 0.25 \
#     --aux-batch-size 256 --aux-warmup-epochs 2.0 \
#     --aux-preflight-every 200 \
#     --aux-abort-flip-rate 0.05 --aux-abort-after-step 500 \
#     --copy-to /content/drive/MyDrive/alphatrain/pillar3c_lam40_best.pt \
#     --save-dir /content/checkpoints/pillar3c_lam40 2>&1 | tee /content/pillar3c_lam40_train.log

In [ ]:
# # Gentler aux: λ=0.10
# %cd /content
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
#     --tensor-file alphatrain/data/v13_pillar3a.pt \
#     --amp --compile \
#     --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
#     --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
#     --target-temperature 0.5 \
#     --aux-counterfactual alphatrain/data/stationary_counterfactuals_v1.pt \
#     --aux-lambda 0.10 --aux-margin 0.25 \
#     --aux-batch-size 256 --aux-warmup-epochs 2.0 \
#     --aux-preflight-every 200 \
#     --aux-abort-flip-rate 0.05 --aux-abort-after-step 500 \
#     --copy-to /content/drive/MyDrive/alphatrain/pillar3c_lam10_best.pt \
#     --save-dir /content/checkpoints/pillar3c_lam10 2>&1 | tee /content/pillar3c_lam10_train.log

## Post-run on M5

After download:

```bash
# Eval each epoch on 1000 OOS seeds 777000-777999 (pillar3b's eval seed range)
for ep in 5 8 11 14 17; do
    python -m alphatrain.scripts.eval_parallel \
        --model alphatrain/data/pillar3c_epoch_${ep}.pt \
        --seeds-range 777000 778000 \
        --device cpu --workers 16 \
        --output alphatrain/data/pillar3c_ep${ep}_eval.json
done
```

**Decision gate** (vs pillar3b: mean 17,255, P5 1,576, <1000 rate 2.5%):

| outcome | criterion |
|---|---|
| Strong win | P5 ≥ 2500 AND <1000 ≤ 1.0% AND mean ≥ 16,400 |
| Acceptable | P5 ≥ 2200 AND <1000 ≤ 1.5% AND mean ≥ 16,400 |
| No-go | mean drops > 10% OR floor doesn't improve |